# Shared Check Conformed

In [0]:
%sql
WITH July26_Period_Dedup AS (
    -- Removing duplicates from period_mat_ytd: removes bimonthly duplicate rows per (LocalPeriodKey, DataProvider, Database)
    SELECT DISTINCT LocalPeriodKey, DataProvider, Database, ytd_flag
    FROM glbl_cpw_prod.adhoc.period_mat_ytd
    WHERE ytd_flag IN ('YTD TY', 'YTD LY')
),
June26_Period_Dedup AS (
    -- Removing Duplicates for June26 period table; restrict to databases still active in current period_mat_ytd
    -- to exclude retired databases that inflate June26 totals
    SELECT DISTINCT m.LocalPeriodKey, m.DataProvider, m.Database, m.ytd_flag
    FROM glbl_cpw_prod.history_conformed.period_mat_ytd_Jun26 m
    WHERE m.ytd_flag IN ('YTD TY', 'YTD LY')
      AND m.Database IN (SELECT DISTINCT Database FROM glbl_cpw_prod.adhoc.period_mat_ytd)
),
-- CTEs for July 2026
July26_RawAggregatedData AS (
    -- Aggregates Value_000_CHF for each Global Manufacturer Type and YTD flag for JJ26.
    SELECT
        gm.Global_Market,
        p.Global_Manufacturer_Type,
        pmy.ytd_flag,
        SUM(f.Value_000 * FX.ExchangeRate) AS Value_000_CHF
    FROM
        glbl_cpw_prod.conformed.factretailsales AS f
    JOIN
        glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN
        glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN
        glbl_cpw_prod.conformed.dimperiod t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN
        July26_Period_Dedup pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
        AND f.DataProvider = pmy.DataProvider
        AND f.Database = pmy.Database
    JOIN
        metadata.forex FX ON f.Database = FX.Database
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y'
        AND P.Global_Category in ('RTE','RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2024', '2025', '2026')
        AND pmy.ytd_flag IN ('YTD TY', 'YTD LY')
        AND gm.Local_Market = gm.Global_Market
    GROUP BY
       gm.Global_Market, -- Group by Global_Market
        p.Global_Manufacturer_Type,
        pmy.ytd_flag
),
July26_TotalsForMarketContext AS ( -- Renamed to reflect per-market totals
    -- Calculates total Value_000_CHF for each Global_Market for JJ26.
    SELECT
        Global_Market, -- Group by market
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_TY,
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_LY,
        COALESCE(SUM(Value_000_CHF), 0) AS Total_Value_000_CHF_Period_Overall
    FROM
        July26_RawAggregatedData
    GROUP BY
        Global_Market -- Group by market
),
July26_ManufacturerShares AS (
    -- Joins aggregated data with market-specific totals for JJ26.
    SELECT
        ad.Global_Market,
        ad.Global_Manufacturer_Type,
        ad.ytd_flag,
        ad.Value_000_CHF,
        tfc.Total_Value_000_CHF_YTD_TY,
        tfc.Total_Value_000_CHF_YTD_LY,
        tfc.Total_Value_000_CHF_Period_Overall
    FROM
        July26_RawAggregatedData ad
    INNER JOIN July26_TotalsForMarketContext tfc ON ad.Global_Market = tfc.Global_Market -- INNER JOIN on Global_Market
),
July26_CalculatedShares AS (
    -- Pivots YTD flags into columns and calculates percentage shares and BPS for JJ26.
    SELECT
        Global_Market, -- Include Global_Market
        Global_Manufacturer_Type,
        (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) AS YTD_TY_Share,
        (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1)) AS YTD_LY_Share,
        (SUM(Value_000_CHF) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_Period_Overall), 0), 1)) AS Grand_Total_Share,
        (
            (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) -
            (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1))
        ) * 100 AS BPS
    FROM
        July26_ManufacturerShares
    GROUP BY
        Global_Market, -- Group by Global_Market
        Global_Manufacturer_Type
),
-- CTEs for June 2026
June26_RawAggregatedData AS (
    -- Aggregates Value_000_CHF for each Global Manufacturer Type and YTD flag for FM26.
    SELECT
        gm.Global_Market,
        p.Global_Manufacturer_Type,
        pmy.ytd_flag,
        SUM(f.Value_000 * FX.ExchangeRate) AS Value_000_CHF
    FROM
        glbl_cpw_prod.history_conformed.factretailsales_Jun26 AS f
    JOIN
        glbl_cpw_prod.history_conformed.dimproduct_Jun26 p ON f.LocalProductKey = p.LocalProductKey
    JOIN
        glbl_cpw_prod.history_conformed.dimmarket_Jun26 gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN
        glbl_cpw_prod.history_conformed.dimperiod_Jun26 t ON f.LocalPeriodKey = t.LocalPeriodKey
    JOIN
        June26_Period_Dedup pmy ON f.LocalPeriodKey = pmy.LocalPeriodKey
        AND f.DataProvider = pmy.DataProvider
        AND f.Database = pmy.Database
    JOIN
        metadata.forex FX ON f.Database = FX.Database
    WHERE
        gm.Global_Total_Mkt_Flag = 'Y'
        AND P.Global_Category in ('RTE','RTE_OTHER')
        AND LEFT(t.Local_Time_Period, 4) IN ('2022', '2023', '2024', '2025', '2026')
        AND pmy.ytd_flag IN ('YTD TY', 'YTD LY')
        AND gm.Local_Market = gm.Global_Market
    GROUP BY
        gm.Global_Market, -- Group by Global_Market
        p.Global_Manufacturer_Type,
        pmy.ytd_flag
),
June26_TotalsForMarketContext AS ( -- Renamed to reflect per-market totals
    -- Calculates total Value_000_CHF for each Global_Market for JJ26.
    SELECT
        Global_Market, -- Group by market
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_TY,
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_LY,
        COALESCE(SUM(Value_000_CHF), 0) AS Total_Value_000_CHF_Period_Overall
    FROM
        June26_RawAggregatedData
    GROUP BY
        Global_Market -- Group by market
),
June26_ManufacturerShares AS (
    -- Joins aggregated data with market-specific totals for JJ26.
    SELECT
        ad.Global_Market,
        ad.Global_Manufacturer_Type,
        ad.ytd_flag,
        ad.Value_000_CHF,
        tfc.Total_Value_000_CHF_YTD_TY,
        tfc.Total_Value_000_CHF_YTD_LY,
        tfc.Total_Value_000_CHF_Period_Overall
    FROM
        June26_RawAggregatedData ad
    INNER JOIN June26_TotalsForMarketContext tfc ON ad.Global_Market = tfc.Global_Market
),
June26_CalculatedShares AS (
    -- Pivots YTD flags into columns and calculates percentage shares and BPS for JJ26.
    SELECT
        Global_Market, -- Include Global_Market
        Global_Manufacturer_Type,
        (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) AS YTD_TY_Share,
        (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1)) AS YTD_LY_Share,
        (SUM(Value_000_CHF) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_Period_Overall), 0), 1)) AS Grand_Total_Share,
        (
            (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) -
            (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1))
        ) * 100 AS BPS
    FROM
        June26_ManufacturerShares
    GROUP BY
        Global_Market, -- Group by Global_Market
        Global_Manufacturer_Type
),
-- Combine and calculate differences
CombinedPivotedData AS (
    SELECT
        COALESCE(apr.Global_Market, mar.Global_Market) AS Global_Market,
        COALESCE(apr.Global_Manufacturer_Type, mar.Global_Manufacturer_Type) AS Global_Manufacturer_Type,
        -- July26 Shares
        COALESCE(apr.YTD_TY_Share, 0) AS July26_YTD_TY_Share,
        COALESCE(apr.YTD_LY_Share, 0) AS July26_YTD_LY_Share,
        COALESCE(apr.Grand_Total_Share, 0) AS July26_Grand_Total_Share,
        COALESCE(apr.BPS, 0) AS July26_BPS,
        -- June26 Shares
        COALESCE(mar.YTD_TY_Share, 0) AS June26_YTD_TY_Share,
        COALESCE(mar.YTD_LY_Share, 0) AS June26_YTD_LY_Share,
        COALESCE(mar.Grand_Total_Share, 0) AS June26_Grand_Total_Share,
        COALESCE(mar.BPS, 0) AS June26_BPS
    FROM
        July26_CalculatedShares apr
    INNER JOIN
        June26_CalculatedShares mar
        ON apr.Global_Manufacturer_Type = mar.Global_Manufacturer_Type
        AND apr.Global_Market = mar.Global_Market -- Join on Global_Market as well
)
-- Final SELECT statement to format the output and add the 'Grand Total' row.
SELECT
    cpd.Global_Market AS `Global_Market`, -- Display Global_Market as a column
    cpd.Global_Manufacturer_Type AS `Global_Manufacturer_Type`,
    -- July26 Metrics
    CONCAT(ROUND(cpd.July26_YTD_TY_Share, 2), '%') AS `July26_YTD TY`,
    CONCAT(ROUND(cpd.July26_YTD_LY_Share, 2), '%') AS `July26_YTD LY`,
    CONCAT(ROUND(cpd.July26_Grand_Total_Share, 2), '%') AS `July26_Grand Total`,
    ROUND(cpd.July26_BPS, 0) AS `July26_BPS`,
 
    -- Difference Metrics
    CONCAT(ROUND(cpd.July26_YTD_TY_Share - cpd.June26_YTD_TY_Share, 2), '%') AS `Diff. Value Share July26 vs June26 YTD TY`,
    CONCAT(ROUND(cpd.July26_YTD_LY_Share - cpd.June26_YTD_LY_Share, 2), '%') AS `Diff. Value Share July26 vs June26 YTD LY`,
    ROUND(cpd.July26_BPS - cpd.June26_BPS, 0) AS `Diff. BPS July26 vs June26`,
 
    -- June26 Metrics
    CONCAT(ROUND(cpd.June26_YTD_TY_Share, 2), '%') AS `June26_YTD TY`,
    CONCAT(ROUND(cpd.June26_YTD_LY_Share, 2), '%') AS `June26_YTD LY`,
    CONCAT(ROUND(cpd.June26_Grand_Total_Share, 2), '%') AS `June26_Grand Total`,
    ROUND(cpd.June26_BPS, 0) AS `June26_BPS`
FROM
    CombinedPivotedData cpd
UNION ALL
-- Grand Total Row
-- This Grand Total row will now represent the overall total across ALL markets and manufacturer types.
SELECT
    'Overall Grand Total' AS `Global_Market`, -- Label for the overall total market
    'Grand Total' AS `Global_Manufacturer_Type`,
 
    -- July26 Grand Total Metrics (should be 100% for shares, 0 for BPS)
    '100.00%' AS `July26_YTD TY`,
    '100.00%' AS `July26_YTD LY`,
    '100.00%' AS `July26_Grand Total`,
    0 AS `July26_BPS`,
 
    -- Difference Grand Total Metrics (should be 0% for shares, 0 for BPS)
    '0.00%' AS `Diff. Value Share July26 vs June26 YTD TY`,
    '0.00%' AS `Diff. Value Share July26 vs June26 YTD LY`,
    0 AS `Diff. BPS July26 vs June26`,
 
    -- July26 Grand Total Metrics (should be 100% for shares, 0 for BPS)
    '100.00%' AS `June26_YTD TY`,
    '100.00%' AS `June26_YTD LY`,
    '100.00%' AS `June26_Grand Total`,
    0 AS `June26_BPS`
FROM
    (SELECT 1) AS dummy -- Dummy table to generate a single row for the UNION ALL
ORDER BY
    CASE WHEN `Global_Market` = 'Overall Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Overall Grand Total' row appears last
    `Global_Market`,
    CASE WHEN `Global_Manufacturer_Type` = 'Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Grand Total' manufacturer type appears last within each market
    `Global_Manufacturer_Type`;

Global_Market,Global_Manufacturer_Type,July26_YTD TY,July26_YTD LY,July26_Grand Total,July26_BPS,Diff. Value Share July26 vs June26 YTD TY,Diff. Value Share July26 vs June26 YTD LY,Diff. BPS July26 vs June26,June26_YTD TY,June26_YTD LY,June26_Grand Total,June26_BPS
Australia,KELLOGGS_MFR TYPE,33.69%,36.84%,35.25%,-315.0,-0.11%,-0.35%,24.0,33.8%,37.19%,35.47%,-339.0
Australia,NESTLE_MFR TYPE,21.48%,21.2%,21.34%,28.0,0.17%,0.21%,-3.0,21.31%,21.0%,21.16%,31.0
Australia,OTHER BRANDED,36.72%,34.6%,35.67%,212.0,-0.06%,0.14%,-20.0,36.78%,34.46%,35.63%,231.0
Australia,PRIVATE LABEL_MFR TYPE,8.11%,7.36%,7.74%,75.0,-0.01%,0.01%,-1.0,8.12%,7.35%,7.74%,77.0
Austria,KELLOGGS_MFR TYPE,20.19%,19.76%,19.97%,43.0,0.08%,0.02%,6.0,20.11%,19.74%,19.92%,37.0
Austria,NESTLE_MFR TYPE,29.89%,30.39%,30.14%,-49.0,0.27%,-0.05%,33.0,29.62%,30.44%,30.03%,-82.0
Austria,OTHER BRANDED,10.34%,11.4%,10.87%,-106.0,-0.2%,0.01%,-21.0,10.54%,11.39%,10.97%,-85.0
Austria,PRIVATE LABEL_MFR TYPE,39.58%,38.46%,39.02%,112.0,-0.15%,0.03%,-18.0,39.73%,38.42%,39.07%,130.0
Austria_Muesli,KELLOGGS_MFR TYPE,2.51%,3.02%,2.76%,-52.0,0.07%,-0.04%,11.0,2.44%,3.07%,2.74%,-63.0
Austria_Muesli,OTHER BRANDED,34.76%,35.69%,35.21%,-93.0,-0.14%,-0.28%,14.0,34.9%,35.96%,35.42%,-107.0


# Shared Check Derived

In [0]:
%sql
-- WITH July26_Period_Dedup AS (
--     -- Removing duplicates from period_mat_ytd: removes bimonthly duplicate rows per (LocalPeriodKey, DataProvider, Database)
--     SELECT DISTINCT LocalPeriodKey, DataProvider, Database, year_month AS YearMonth, ytd_flag AS YTD_Flag
--     FROM glbl_cpw_prod.adhoc.period_mat_ytd
--     WHERE ytd_flag IN ('YTD TY', 'YTD LY')
-- ),
-- CTEs for July 2026
With July26_RawAggregatedData AS (
    -- Aggregates Value_000_CHF for each Manufacturer_type and YTD flag for AM26.
    SELECT
        Market AS Global_Market,
        Manufacturer_type AS Global_Manufacturer_Type,
        -- Global_SKU,
        YTD_Flag AS ytd_flag,
        SUM(Value_000_CHF) AS Value_000_CHF
    FROM
        glbl_cpw_prod.derived.retailindextotal r
    WHERE
       -- r.Market_Agg = r.Market AND
        r.Category in ('RTE','RTE_OTHER')
        AND LEFT(r.YearMonth, 4) IN ('2022', '2023', '2024', '2025', '2026')
        AND r.YTD_Flag IN ('YTD TY', 'YTD LY')
        AND r.Market NOT IN ('Colombia', 'Costa Rica', 'El Salvador','Estonia','Latvia','Lithuania','Guatemala','Greece_CC','Honduras','Nicaragua', 'Panamá', 'Peru')
  GROUP BY
    r.Market,
    r.Manufacturer_type,
    r.YTD_Flag
),
July26_TotalsForMarketContext AS (
    -- Calculates total Value_000_CHF for each Global_Market for AM26.
    SELECT
        Global_Market,
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_TY,
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_LY,
        COALESCE(SUM(Value_000_CHF), 0) AS Total_Value_000_CHF_Period_Overall
    FROM
        July26_RawAggregatedData
    GROUP BY
        Global_Market
),
July26_ManufacturerShares AS (
    -- Joins aggregated data with market-specific totals for AM26.
    SELECT
        ad.Global_Market,
        ad.Global_Manufacturer_Type,
        ad.ytd_flag,
        ad.Value_000_CHF,
        tfc.Total_Value_000_CHF_YTD_TY,
        tfc.Total_Value_000_CHF_YTD_LY,
        tfc.Total_Value_000_CHF_Period_Overall
    FROM
        July26_RawAggregatedData ad
    INNER JOIN July26_TotalsForMarketContext tfc ON ad.Global_Market = tfc.Global_Market
),
July26_CalculatedShares AS (
    -- Pivots YTD flags into columns and calculates percentage shares and BPS for AM26.
    SELECT
        Global_Market,
        Global_Manufacturer_Type,
        -- Global_SKU,
        (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) AS YTD_TY_Share,
        (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1)) AS YTD_LY_Share,
        (SUM(Value_000_CHF) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_Period_Overall), 0), 1)) AS Grand_Total_Share,
        (
            (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) -
            (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1))
        ) * 100 AS BPS
    FROM
        July26_ManufacturerShares
    GROUP BY
        Global_Market
        ,Global_Manufacturer_Type
        -- ,Global_SKU
),
-- CTEs for June 2026
Jun26_RawAggregatedData AS (
  -- Aggregates Value_000_CHF for each Manufacturer_type and YTD flag for FM26.
  SELECT
    r.Market AS Global_Market,
    r.Manufacturer_type AS Global_Manufacturer_Type,
    r.YTD_Flag AS ytd_flag,
    SUM(r.Value_000_CHF) AS Value_000_CHF
  FROM
    glbl_cpw_prod.history_conformed.retailindextotal_Jun26 r
  WHERE
    r.Category in ('RTE','RTE_OTHER')
    AND LEFT(r.YearMonth, 4) IN ('2022', '2023', '2024', '2025', '2026')
    AND r.YTD_Flag IN ('YTD TY', 'YTD LY')
    AND r.Market NOT IN ('Colombia', 'Costa Rica', 'El Salvador','Estonia','Latvia','Lithuania','Guatemala','Greece_CC','Honduras','Nicaragua', 'Panamá', 'Peru')
  GROUP BY
    r.Market,
    r.Manufacturer_type,
    r.YTD_Flag
),
Jun26_TotalsForMarketContext AS ( -- Renamed to reflect per-market totals
    -- Calculates total Value_000_CHF for each Global_Market for FM26.
    SELECT
        Global_Market, -- Group by market
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_TY,
        COALESCE(SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END), 0) AS Total_Value_000_CHF_YTD_LY,
        COALESCE(SUM(Value_000_CHF), 0) AS Total_Value_000_CHF_Period_Overall
    FROM
        Jun26_RawAggregatedData
    GROUP BY
        Global_Market -- Group by market
),
Jun26_ManufacturerShares AS (
  -- Joins aggregated data with market-specific totals for FM26.
  SELECT
    ad.Global_Market,
    ad.Global_Manufacturer_Type,
    ad.ytd_flag,
    ad.Value_000_CHF,
    tfc.Total_Value_000_CHF_YTD_TY,
    tfc.Total_Value_000_CHF_YTD_LY,
    tfc.Total_Value_000_CHF_Period_Overall
  FROM
    Jun26_RawAggregatedData ad
      INNER JOIN Jun26_TotalsForMarketContext tfc
        ON ad.Global_Market = tfc.Global_Market
),
Jun26_CalculatedShares AS (
    -- Pivots YTD flags into columns and calculates percentage shares and BPS for FM26.
    SELECT
        Global_Market, -- Include Global_Market
        Global_Manufacturer_Type,
        (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) AS YTD_TY_Share,
        (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1)) AS YTD_LY_Share,
        (SUM(Value_000_CHF) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_Period_Overall), 0), 1)) AS Grand_Total_Share,
        (
            (SUM(CASE WHEN ytd_flag = 'YTD TY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_TY), 0), 1)) -
            (SUM(CASE WHEN ytd_flag = 'YTD LY' THEN Value_000_CHF ELSE 0 END) * 100.0 / COALESCE(NULLIF(MAX(Total_Value_000_CHF_YTD_LY), 0), 1))
        ) * 100 AS BPS
    FROM
        Jun26_ManufacturerShares
    GROUP BY
        Global_Market, -- Group by Global_Market
        Global_Manufacturer_Type
),
-- Combine and calculate differences
CombinedPivotedData AS (
  SELECT
    COALESCE(apr.Global_Market, mar.Global_Market) AS Global_Market,
    COALESCE(
      apr.Global_Manufacturer_Type,
      mar.Global_Manufacturer_Type
    ) AS Global_Manufacturer_Type,
    -- July26 Shares
    COALESCE(apr.YTD_TY_Share, 0) AS July26_YTD_TY_Share,
    COALESCE(apr.YTD_LY_Share, 0) AS July26_YTD_LY_Share,
    COALESCE(apr.Grand_Total_Share, 0) AS July26_Grand_Total_Share,
    COALESCE(apr.BPS, 0) AS July26_BPS,
    -- Jun26 Shares
    COALESCE(mar.YTD_TY_Share, 0) AS Jun26_YTD_TY_Share,
    COALESCE(mar.YTD_LY_Share, 0) AS Jun26_YTD_LY_Share,
    COALESCE(mar.Grand_Total_Share, 0) AS Jun26_Grand_Total_Share,
    COALESCE(mar.BPS, 0) AS Jun26_BPS
  FROM
    July26_CalculatedShares apr
      INNER JOIN Jun26_CalculatedShares mar
        ON apr.Global_Manufacturer_Type = mar.Global_Manufacturer_Type
        AND apr.Global_Market = mar.Global_Market
)
-- Final SELECT statement to format the output and add the 'Grand Total' row.
SELECT
    cpd.Global_Market AS `Global_Market`, -- Display Global_Market as a column
    cpd.Global_Manufacturer_Type AS `Global_Manufacturer_Type`,
    -- July26 Metrics
    CONCAT(ROUND(cpd.July26_YTD_TY_Share, 2), '%') AS `July26_YTD TY`,
    CONCAT(ROUND(cpd.July26_YTD_LY_Share, 2), '%') AS `July26_YTD LY`,
    CONCAT(ROUND(cpd.July26_Grand_Total_Share, 2), '%') AS `July26_Grand Total`,
    ROUND(cpd.July26_BPS, 0) AS `July26_BPS`,
 
    -- Difference Metrics
    CONCAT(ROUND(cpd.July26_YTD_TY_Share - cpd.Jun26_YTD_TY_Share, 2), '%') AS `Diff. Value Share July26 vs Jun26 YTD TY`,
    CONCAT(ROUND(cpd.July26_YTD_LY_Share - cpd.Jun26_YTD_LY_Share, 2), '%') AS `Diff. Value Share July26 vs Jun26 YTD LY`,
    ROUND(cpd.July26_BPS - cpd.Jun26_BPS, 0) AS `Diff. BPS July26 vs Jun26`,
 
    -- Jun26 Metrics
    CONCAT(ROUND(cpd.Jun26_YTD_TY_Share, 2), '%') AS `Jun26_YTD TY`,
    CONCAT(ROUND(cpd.Jun26_YTD_LY_Share, 2), '%') AS `Jun26_YTD LY`,
    CONCAT(ROUND(cpd.Jun26_Grand_Total_Share, 2), '%') AS `Jun26_Grand Total`,
    ROUND(cpd.Jun26_BPS, 0) AS `Jun26_BPS`
FROM
    CombinedPivotedData cpd
UNION ALL
-- Grand Total Row
-- This Grand Total row will now represent the overall total across ALL markets and manufacturer types.
SELECT
    'Overall Grand Total' AS `Global_Market`, -- Label for the overall total market
    'Grand Total' AS `Global_Manufacturer_Type`,
 
    -- July26 Grand Total Metrics (should be 100% for shares, 0 for BPS)
    '100.00%' AS `July26_YTD TY`,
    '100.00%' AS `July26_YTD LY`,
    '100.00%' AS `July26_Grand Total`,
    0 AS `July26_BPS`,
 
    -- Difference Grand Total Metrics (should be 0% for shares, 0 for BPS)
    '0.00%' AS `Diff. Value Share July26 vs Jun26 YTD TY`,
    '0.00%' AS `Diff. Value Share July26 vs Jun26 YTD LY`,
    0 AS `Diff. BPS July26 vs Jun26`,
 
    -- Jun26 Grand Total Metrics (should be 100% for shares, 0 for BPS)
    '100.00%' AS `Jun26_YTD TY`,
    '100.00%' AS `Jun26_YTD LY`,
    '100.00%' AS `Jun26_Grand Total`,
    0 AS `Jun26_BPS`
FROM
    (SELECT 1) AS dummy -- Dummy table to generate a single row for the UNION ALL
ORDER BY
    CASE WHEN `Global_Market` = 'Overall Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Overall Grand Total' row appears last
    `Global_Market`,
    CASE WHEN `Global_Manufacturer_Type` = 'Grand Total' THEN 2 ELSE 1 END, -- Ensures 'Grand Total' manufacturer type appears last within each market
    `Global_Manufacturer_Type`;

Global_Market,Global_Manufacturer_Type,July26_YTD TY,July26_YTD LY,July26_Grand Total,July26_BPS,Diff. Value Share July26 vs Jun26 YTD TY,Diff. Value Share July26 vs Jun26 YTD LY,Diff. BPS July26 vs Jun26,Jun26_YTD TY,Jun26_YTD LY,Jun26_Grand Total,Jun26_BPS
Australia,KELLOGGS_MFR TYPE,33.76%,36.91%,35.31%,-315,-0.11%,-0.35%,24,33.87%,37.26%,35.54%,-340
Australia,NESTLE_MFR TYPE,21.52%,21.25%,21.39%,28,0.17%,0.21%,-4,21.35%,21.04%,21.20%,31
Australia,OTHER BRANDED,36.63%,34.54%,35.60%,209,-0.06%,0.14%,-20,36.69%,34.40%,35.56%,229
Australia,PRIVATE LABEL_MFR TYPE,8.09%,7.31%,7.70%,78,0.00%,0.01%,-1,8.09%,7.30%,7.70%,79
Austria,KELLOGGS_MFR TYPE,20.19%,19.76%,19.97%,43,0.08%,0.01%,6,20.11%,19.75%,19.93%,36
Austria,NESTLE_MFR TYPE,29.89%,30.40%,30.14%,-50,0.27%,-0.06%,33,29.62%,30.45%,30.04%,-83
Austria,OTHER BRANDED,10.34%,11.37%,10.86%,-103,-0.20%,0.01%,-21,10.54%,11.36%,10.95%,-82
Austria,PRIVATE LABEL_MFR TYPE,39.58%,38.47%,39.02%,111,-0.15%,0.03%,-18,39.73%,38.44%,39.08%,129
Austria_Muesli,KELLOGGS_MFR TYPE,2.73%,3.28%,3.00%,-55,0.07%,-0.05%,12,2.66%,3.33%,2.99%,-67
Austria_Muesli,OTHER BRANDED,33.28%,33.39%,33.34%,-11,-0.18%,-0.24%,6,33.46%,33.63%,33.55%,-17


# Global Manufacturer Type Product vs Product History

In [0]:
%sql
WITH reclass AS (
    -- Products whose Manufacturer Type changed
    SELECT p_cur.product_id, p_cur.Local_Market,
           p_old.Global_Manufacturer_Type AS seg_jun, p_cur.Global_Manufacturer_Type AS seg_jul
    FROM glbl_cpw_prod.conformed.dimproduct p_cur
    JOIN glbl_cpw_prod.history_conformed.dimproduct_Jun26 p_old
      ON p_cur.product_id = p_old.product_id AND p_cur.Local_Market = p_old.Local_Market
    WHERE p_cur.Global_Category IN ('RTE','RTE_OTHER')
      AND p_cur.Local_Market IN ('Spain','Switzerland')
      AND p_cur.Global_Manufacturer_Type <> p_old.Global_Manufacturer_Type
),
jul26_sales AS (
    -- Current conformed sales for the reclassified products (MAT scope)
    SELECT p.Local_Market, p.product_id, p.Global_Manufacturer_Type,
           SUM(f.Value_000) AS Value_000, SUM(f.Volume_000) AS Volume_000, p.Global_Manufacturer_Type
    FROM glbl_cpw_prod.conformed.factretailsales f
    JOIN glbl_cpw_prod.conformed.dimproduct p ON f.LocalProductKey = p.LocalProductKey
    JOIN glbl_cpw_prod.conformed.dimmarket gm ON f.LocalMarketKey = gm.LocalMarketKey
    JOIN glbl_cpw_prod.adhoc.period_mat_ytd pmt
      ON f.LocalPeriodKey = pmt.LocalPeriodKey AND f.DataProvider = pmt.DataProvider AND f.Database = pmt.Database
    WHERE gm.Global_Total_Mkt_Flag = 'Y'
      AND p.Global_Category IN ('RTE','RTE_OTHER')
      AND p.Local_Market IN ('Spain','Switzerland')
      AND pmt.MAT_Flag IN ('MAT TY','MAT LY','MAT 2LY')
    --   AND p.Global_Manufacturer_Type <> p.Global_Manufacturer_Type
    GROUP BY p.Local_Market, p.product_id, p.Global_OGF_Segment, p.Global_Manufacturer_Type
),
market_totals AS (
    SELECT Local_Market, SUM(Value_000) AS Total_Value, SUM(Volume_000) AS Total_Volume
    FROM jul26_sales GROUP BY Local_Market
)
SELECT r.Local_Market, r.product_id, r.seg_jun, r.seg_jul,
       ROUND(s.Value_000, 2) AS Value_000,
       ROUND(s.Value_000 * 100.0 / mt.Total_Value, 2) AS Pct_of_Market_Value,
       ROUND(s.Volume_000, 2) AS Volume_000,
       ROUND(s.Volume_000 * 100.0 / mt.Total_Volume, 2) AS Pct_of_Market_Volume
FROM reclass r
LEFT JOIN jul26_sales s ON r.product_id = s.product_id AND r.Local_Market = s.Local_Market
LEFT JOIN market_totals mt ON r.Local_Market = mt.Local_Market
ORDER BY r.Local_Market, r.product_id

Local_Market,product_id,seg_jun,seg_jul,Value_000,Pct_of_Market_Value,Volume_000,Pct_of_Market_Volume
